# Bringing Your Own Dataset into RadHarmony

Step-by-step guide and copy-paste template for integrating a new radiological dataset.

---

## Architecture

Every dataset is built from two pieces:

| Layer | Class | Your job |
|---|---|---|
| **Harmonizer** | `YourHarmonizer(BaseHarmonizer)` | Read raw CSVs and return a standardized DataFrame |
| **Dataset** | `YourDataset(BaseRadiologicalDataset)` | Declare `LABEL_COLS`, set `_HARMONIZER_CLS`, and wire `_get_harmonized_df` (with optional preset resolution) |

The base class handles MONAI transforms, `PersistentDataset` caching, patient-level splitting, k-fold CV, and pre-caching. Decorating with `@register_dataset("name")` adds it to the global registry. Set **`_HARMONIZER_CLS`** on the dataset class so **`harmonizer_path=`** can reload pickles from **`BaseHarmonizer.save`**.

---

## Steps

| Step | |
|---|---|
| 1 | Write the Harmonizer |
| 2 | Write the Dataset class |
| 3 | Validate the harmonized DataFrame |
| 4 | Split, inspect, and visualize |
| 5 | Augmentations and custom transforms |
| 6 | Use the registry |
| 7 | Add to the Gradio app |

---

## What you need before starting

| Item | Example |
|---|---|
| Image root directory | `/data/my_dataset/images/` |
| CSV(s) with labels / metadata | `labels.csv` |
| How image paths are encoded | `dicom_id` column, or absolute `image_path` |
| Patient-level identifier | `PatientID`, `subject_id` |
| Study-level identifier | `StudyInstanceUID`, `accession_number` |
| Label columns | `["Pneumonia", "Pleural Effusion"]` |
| Modality | 2D (X-ray/DICOM) or 3D (CT/NIfTI) |


---

## Step 1 — Write the Harmonizer

Subclass `BaseHarmonizer`. Implement three abstract methods and configure class-level attributes for optional columns:

| Method | What to implement |
|---|---|
| `_build_patient_id()` | Set `self.df["patient_id"]` |
| `_build_study_id()` | Set `self.df["study_id"]` |
| `_build_image_path()` | Set `self.df["image_path"]` with paths **relative to `base_image_dir`** |

**`base_image_dir` convention** — `base_image_dir` must point at the **deepest stable directory whose immediate contents are the entries named in `image_path`**. Don't bake a hardcoded subdirectory prefix into `image_path` that could be folded into `base_image_dir` instead. For Train/Test splits, each variant takes its own split-specific directory (e.g. `train_images/`, `test_images/`). See `.claude/skills/add-dataset/SKILL.md` for the full convention and built-in examples (CheXpert, MIMIC-CXR, RSNA PE Detection, …).

| Attribute | Purpose |
|---|---|
| `LABEL_COLS` | Raw label column names (auto snake_cased) |
| `LABEL_JOIN_COLS` | Join keys for a separate label CSV |
| `VIEW_POSITION_SOURCE_COL` | Column to rename → `view_position` |
| `VIEW_POSITION_JOIN_COLS` | Join keys for a separate view-position CSV |
| `MASK_JOIN_COLS` / `MASK_SOURCE_COL` | Join keys and source column for mask CSV |
| `BBOX_JOIN_COLS` / `BBOX_SOURCE_COL` | Join keys and source column for bbox CSV |
| `REPORT_JOIN_COLS` / `REPORT_PATH_COL` | Join keys and file-path column for report CSV |

Optional preprocessing hooks (override to clean each CSV before it is merged):
`_preprocess_label_df`, `_preprocess_view_position_df`, `_preprocess_mask_df`, `_preprocess_bbox_df`, `_preprocess_report_df`, `_decode_mask`

If your harmonizer adds `__init__` parameters beyond `BaseHarmonizer` (e.g. `base_image_dir`), override **`_harmonizer_init_snapshot()`** and merge in those keys so **`save()`** / **`load_from_saved()`** can rebuild the instance (see built-in harmonizers for examples).

In [ ]:
import os
import numpy as np
import pandas as pd

from radharmony.harmonizer.base import BaseHarmonizer


class MyDatasetHarmonizer(BaseHarmonizer):  # MUST CHANGE: rename
    """Harmonize MyDataset into the standard RadHarmony format."""

    # MUST CHANGE: raw column names as they appear in the CSV.
    # BaseHarmonizer snake_cases them (e.g. "Pleural Effusion" → "pleural_effusion").
    LABEL_COLS = ["Pneumonia", "Pleural Effusion"]
    LABEL_JOIN_COLS = ["patient_id", "study_id"]  # join keys for a separate label CSV

    VIEW_POSITION_SOURCE_COL = "ViewPosition"  # rename to "view_position"; or None
    VIEW_POSITION_JOIN_COLS = ["patient_id", "study_id"]

    MASK_SOURCE_COL = "mask_path"  # rename to "mask_path"; or None
    MASK_JOIN_COLS = ["patient_id", "study_id"]

    BBOX_SOURCE_COL = "bbox"  # rename to "bbox"; or None
    BBOX_JOIN_COLS = ["patient_id", "study_id"]

    REPORT_PATH_COL = "report_path"  # column containing report file paths; or None
    REPORT_JOIN_COLS = ["patient_id", "study_id"]

    def __init__(
        self,
        csv_path: str,
        label_csv_path: str = None,
        base_image_dir: str = None,
    ):
        super().__init__(csv_path=csv_path, label_csv_path=label_csv_path)
        # base_image_dir is stored so the dataset class can join it with the
        # relative image_path values at load time. Some harmonizers also use
        # it inside _build_image_path (e.g. to scan for files); see ChestX-ray14.
        self.base_image_dir = base_image_dir

    def _harmonizer_init_snapshot(self) -> dict:
        # Required for save() / load_from_saved() to round-trip extra params.
        snap = super()._harmonizer_init_snapshot()
        snap["base_image_dir"] = self.base_image_dir
        return snap

    # ------------------------------------------------------------------
    # Abstract methods — MUST implement all three
    # ------------------------------------------------------------------

    def _build_patient_id(self) -> None:
        """Set self.df["patient_id"]."""
        self.df["patient_id"] = self.df["PatientID"].astype(str)  # MUST CHANGE

    def _build_study_id(self) -> None:
        """Set self.df["study_id"]."""
        self.df["study_id"] = self.df["StudyInstanceUID"].astype(str)  # MUST CHANGE

    def _build_image_path(self) -> None:
        """Set self.df["image_path"] with paths RELATIVE to base_image_dir.

        Do NOT include a hardcoded root-level subdirectory prefix that could
        be folded into base_image_dir instead. The dataset class joins
        base_image_dir + image_path at sample-load time.
        """
        # MUST CHANGE: adjust folder structure and file extension.
        # Example: image_path = "<patient>/<study>/<dicom_id>.dcm"
        self.df["image_path"] = self.df.apply(
            lambda r: os.path.join(
                str(r["patient_id"]),
                str(r["study_id"]),
                r["dicom_id"] + ".dcm",  # MUST CHANGE
            ),
            axis=1,
        )

    def harmonize(self, *args, drop_uncertain: bool = True, **kwargs):
        self.drop_uncertain = drop_uncertain
        return super().harmonize(*args, **kwargs)

    # ------------------------------------------------------------------
    # Optional hooks — override those that apply, delete the rest
    # ------------------------------------------------------------------

    def _preprocess_label_df(self, label_df: pd.DataFrame) -> pd.DataFrame:
        """Clean the label CSV before it is merged. Common tasks: fill NaNs,
        drop uncertain labels (-1), cast join column types."""
        label_df = label_df.fillna(0.0)
        label_df = label_df.replace({-1: np.nan})
        if self.drop_uncertain:
            label_df = label_df.dropna()
        label_df["patient_id"] = label_df["patient_id"].astype(str)
        label_df["study_id"] = label_df["study_id"].astype(str)
        return label_df

    def _preprocess_view_position_df(self, vp_df: pd.DataFrame) -> pd.DataFrame:
        """Clean the view-position CSV before it is merged."""
        vp_df["patient_id"] = vp_df["patient_id"].astype(str)
        vp_df["study_id"] = vp_df["study_id"].astype(str)
        return vp_df

    def _preprocess_mask_df(self, mask_df: pd.DataFrame) -> pd.DataFrame:
        """Clean the mask CSV before it is merged."""
        mask_df["patient_id"] = mask_df["patient_id"].astype(str)
        mask_df["study_id"] = mask_df["study_id"].astype(str)
        return mask_df

    def _preprocess_bbox_df(self, bbox_df: pd.DataFrame) -> pd.DataFrame:
        """Clean the bbox CSV before it is merged."""
        bbox_df["patient_id"] = bbox_df["patient_id"].astype(str)
        bbox_df["study_id"] = bbox_df["study_id"].astype(str)
        return bbox_df

    def _preprocess_report_df(self, report_df: pd.DataFrame) -> pd.DataFrame:
        """Clean the report CSV before it is merged."""
        report_df["patient_id"] = report_df["patient_id"].astype(str)
        report_df["study_id"] = report_df["study_id"].astype(str)
        return report_df

    def _decode_mask(self, rle: str, width: int, height: int) -> np.ndarray:
        """Decode an encoded mask string to a (H, W) uint8 array.

        Only needed when mask_path holds encoded strings (e.g. RLE) rather than
        file paths. Return None to skip a row. Called by preprocess_masks().
        """
        # Example: return rle_decode(rle, width, height)
        return None

---

## Step 2 — Write the Dataset class

Subclass `BaseRadiologicalDataset` and implement:

1. **`LABEL_COLS`** — snake_cased label names matching the harmonized DataFrame.
2. **`_HARMONIZER_CLS`** — set this to your harmonizer class (e.g. `_HARMONIZER_CLS = MyDatasetHarmonizer`). Required for **`harmonizer_path=`** so the base can call `YourHarmonizer.load_from_saved(path)`; optional if you only harmonize on the fly.
3. **`_get_harmonized_df()`** — call **`_try_resolve_preset_harmonized()`** first; if it returns a DataFrame, return it. Otherwise build your harmonizer and **`harmonize()`** as usual.

Forward **`harmonized_df`**, **`harmonizer`**, and **`harmonizer_path`** from your constructor into **`super().__init__(...)`** so users can skip slow harmonization by passing a pre-built table, an in-memory harmonizer, or a path to **`save()`** output.

Everything else (transforms, caching, splitting) is inherited.

In [ ]:
import pandas as pd
from radharmony.dataset import (
    BaseRadiologicalDataset,
    RadiologyTransform2D,
    RadiologyTransform3D,
)
from radharmony.registry import register_dataset


@register_dataset("my_dataset")  # MUST CHANGE: unique registry name
class MyDataset(BaseRadiologicalDataset):  # MUST CHANGE: rename
    """PyTorch dataset for MyDataset.

    Args:
        base_image_dir: Deepest stable directory whose immediate contents
            are the entries named in image_path. The base joins each row's
            relative image_path at sample-load time.
        csv_path: Path to the primary metadata / labels CSV.
        label_csv_path: Path to a separate label CSV (optional).
        transform: MONAI Compose transform. Defaults to RadiologyTransform2D.
        cache_dir: PersistentDataset cache directory. None disables caching.
        output_cls: Yield label vector under key ``cls``.
        output_mask: Yield segmentation mask under key ``mask``.
        output_report: Yield report path string under key ``report``.
        output_bbox: Yield bounding box under key ``bbox``.
        drop_uncertain: If ``True`` (default), rows with uncertain labels are dropped.
        harmonized_df: Optional pre-built harmonized DataFrame (skips harmonize).
        harmonizer: Optional harmonizer instance with ``harmonized_df`` set (e.g. after ``harmonize()`` or ``load_from_saved``).
        harmonizer_path: Path to a pickle from ``BaseHarmonizer.save`` (requires ``_HARMONIZER_CLS``).
    """

    # MUST CHANGE: snake_cased label names returned by the harmonizer.
    LABEL_COLS = [
        "pneumonia",
        "pleural_effusion",
    ]

    # MUST CHANGE: same class as in Step 1 — needed for harmonizer_path= / load_from_saved.
    _HARMONIZER_CLS = MyDatasetHarmonizer

    def __init__(
        self,
        base_image_dir: str,
        csv_path: str,
        label_csv_path: str = None,
        transform=None,
        cache_dir: str = "./cache",
        output_cls: bool = False,
        output_mask: bool = False,
        output_report: bool = False,
        output_bbox: bool = False,
        drop_uncertain: bool = True,
        harmonized_df=None,
        harmonizer=None,
        harmonizer_path: str = None,
    ):
        self._csv_path = csv_path
        self._label_csv_path = label_csv_path
        self._drop_uncertain = drop_uncertain

        output_keys = {"img"}
        if output_cls:
            output_keys.add("cls")
        if output_mask:
            output_keys.add("mask")
        if output_report:
            output_keys.add("report")
        if output_bbox:
            output_keys.add("bbox")

        # Use RadiologyTransform3D for CT volumes.
        super().__init__(
            base_image_dir=base_image_dir,
            transform=transform
            or RadiologyTransform2D(
                img_size=224, output_keys=output_keys
            ).get_transform(),
            cache_dir=cache_dir,
            output_cls=output_cls,
            output_mask=output_mask,
            output_report=output_report,
            output_bbox=output_bbox,
            harmonized_df=harmonized_df,
            harmonizer=harmonizer,
            harmonizer_path=harmonizer_path,
        )

    def _get_harmonized_df(self) -> pd.DataFrame:
        """Return the standardized DataFrame for this dataset."""
        preset = self._try_resolve_preset_harmonized()
        if preset is not None:
            return preset
        # MUST CHANGE: add any extra CSV paths your harmonizer needs.
        return MyDatasetHarmonizer(
            csv_path=self._csv_path,
            label_csv_path=self._label_csv_path,
            base_image_dir=self.base_image_dir,
        ).harmonize(drop_uncertain=self._drop_uncertain)

---

## Step 3 — Validate the harmonized DataFrame

Check the harmonizer output before creating any datasets.

In [ ]:
# MUST CHANGE: set your actual paths
BASE_IMAGE_DIR = "/path/to/my_dataset/images"
CSV_PATH = "/path/to/my_dataset/labels.csv"
LABEL_CSV_PATH = None  # or "/path/to/my_dataset/labels_split.csv"
CACHE_DIR = "./cache/my_dataset"

df = MyDatasetHarmonizer(
    csv_path=CSV_PATH,
    label_csv_path=LABEL_CSV_PATH,
    base_image_dir=BASE_IMAGE_DIR,
).harmonize()

print(f"Shape          : {df.shape}")
print(f"Unique patients: {df['patient_id'].nunique():,}")
print(f"Unique studies : {df['study_id'].nunique():,}\n")

# Required columns
missing = [c for c in ["patient_id", "study_id", "image_path"] if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"
print("Required columns: OK")

# image_path should be RELATIVE (no leading "/" or drive letter).
abs_count = df["image_path"].str.startswith("/").sum()
assert abs_count == 0, (
    f"{abs_count} image_path values are absolute. The convention is to store "
    "paths relative to base_image_dir; the dataset class joins them at runtime."
)
print("image_path is relative: OK")

# Label range [0, 1]
for col in MyDataset.LABEL_COLS:
    assert df[col].between(0, 1).all(), f"Label '{col}' has values outside [0, 1]"
print(f"Label range [0,1]: OK  {MyDataset.LABEL_COLS}")

# Spot-check image paths (5 samples) — join base_image_dir with the relative path.
for rel in df.sample(min(5, len(df)), random_state=0)["image_path"]:
    full = os.path.join(BASE_IMAGE_DIR, rel)
    assert os.path.exists(full), f"Image not found: {full}"
print("Image paths exist (5 sampled): OK\n")

# Label prevalence
print("Label prevalence:")
print(df[MyDataset.LABEL_COLS].mean().round(3).to_string())

df.head(3)

---

## Step 4 — Basic dataset usage

### 4.1 Train / validation split

`get_datasets(n_splits=N)` returns `(train_dataset, val_dataset)` with an ~`(N-1)/N` / `1/N` split. Splitting is **patient-level** — no patient appears in both sets.

In [ ]:
ds = MyDataset(
    base_image_dir=BASE_IMAGE_DIR,
    csv_path=CSV_PATH,
    label_csv_path=LABEL_CSV_PATH,
    cache_dir=CACHE_DIR,
    output_cls=True,
)

train_ds, val_ds = ds.get_datasets(n_splits=10, num_cores=4)

print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")

### 4.2 Inspect a sample

Each sample is a dict. Keys present depend on the `output_*` flags enabled.

In [ ]:
import torch

sample = train_ds[0]

for key, val in sample.items():
    if isinstance(val, torch.Tensor):
        print(
            f"  {key:12s}  shape={tuple(val.shape)}  dtype={val.dtype}  "
            f"min={val.float().min():.3f}  max={val.float().max():.3f}"
        )
    else:
        print(f"  {key:12s}  {type(val).__name__}: {val}")

### 4.3 Visualize a sample

In [ ]:
import matplotlib.pyplot as plt

sample = train_ds[0]
img = sample["img"].float().squeeze().numpy()  # (H, W) for 2D

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(img, cmap="gray", vmin=-1, vmax=1)
ax.axis("off")

if "cls" in sample:
    pos = [MyDataset.LABEL_COLS[i] for i, v in enumerate(sample["cls"]) if v > 0]
    ax.set_title("  ".join(pos) if pos else "no finding", fontsize=8)

plt.tight_layout()
plt.show()

---

## Step 5 — Advanced usage

### 5.1 Augmentations — builder API

`RadiologyTransform2D` / `RadiologyTransform3D` expose a `with_*` API for layering augmentations on top of the standard preprocessing pipeline:

| Method | What it does |
|---|---|
| `.with_flip(spatial_axis, prob)` | Random flip — discrete, no interpolation |
| `.with_affine(translate_range, rotate_range, scale_range, shear_range, prob)` | Translate + rotate + scale + shear in one interpolation step |
| `.with_intensity_jitter(shift, scale, prob)` | Random brightness / contrast shift |
| `.with_gaussian_noise(mean, std, prob)` | Additive Gaussian noise |
| `.with_gaussian_smooth(sigma_range, prob)` | Random Gaussian blur |
| `.with_elastic_deformation(magnitude_range, spacing, prob)` | Grid-based elastic warp |
| `.with_bbox_as_mask(keep_mask)` | Convert bbox to a binary spatial mask (`bbox_mask` key) |
| `.with_transpose(indices, keys)` | Reorder spatial axes |

**Example order:** `with_flip` → `with_affine` → `with_intensity_jitter` → `with_gaussian_noise` → `with_gaussian_smooth` → `with_elastic_deformation`

Use `.add_transform(t)` to insert any MONAI transform not covered above.

---

### 5.2 Custom transforms — raw MONAI Compose

Pass a `monai.transforms.Compose` to override the entire pipeline. The keys present in each sample dict depend on the `output_*` flags:

| Key | Enabled by | Content |
|---|---|---|
| `"img"` | always | image tensor |
| `"cls"` | `output_cls=True` | float label vector, `len(LABEL_COLS)` |
| `"mask"` | `output_mask=True` | binary mask tensor |
| `"report"` | `output_report=True` | report file path string |
| `"bbox"` | `output_bbox=True` | bounding box from the `bbox` column |

**Important:** every key listed in a transform's `keys` argument must be present in the dict — MONAI raises a `KeyError` otherwise. Edit `ToTensorD` and `SelectItemsD` to match exactly the `output_*` flags you enable.

In [ ]:
from radharmony.dataset import RadiologyTransform2D, RadiologyTransform3D

# ---- 2-D example: X-ray / DICOM -----------------------------------------------
transform_2d = (
    RadiologyTransform2D(img_size=224, output_keys={"img", "cls"})
    .with_flip(spatial_axis=1, prob=0.5)  # left-right flip
    .with_affine(  # spatial warp — one interpolation step
        translate_range=(10, 10),
        rotate_range=(0.17,),  # ≈ 10°
        scale_range=(0.1, 0.1),  # zoom ±10 %
        prob=0.5,
    )
    .with_intensity_jitter(shift=0.1, scale=0.1, prob=0.5)  # brightness / contrast
    .with_gaussian_noise(std=0.05, prob=0.3)
    .get_transform()
)

# ---- 3-D example: CT volume ----------------------------------------------------
transform_3d = (
    RadiologyTransform3D(
        img_size=112, hu_window=(-1000, 400), output_keys={"img", "cls"}
    )
    .with_flip(spatial_axis=0, prob=0.5)
    .with_affine(
        translate_range=(5, 5, 5),
        rotate_range=(0.09, 0.09, 0.09),  # ≈ 5° per axis
        scale_range=(0.1, 0.1, 0.1),
        prob=0.5,
    )
    .with_intensity_jitter(shift=0.05, scale=0.05, prob=0.5)
    .get_transform()
)

# Pass the transform to the dataset constructor
ds = MyDataset(
    base_image_dir=BASE_IMAGE_DIR,
    csv_path=CSV_PATH,
    transform=transform_2d,
    cache_dir=CACHE_DIR,
    output_cls=True,
)
train_ds, val_ds = ds.get_datasets(n_splits=10, num_cores=4)
print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")

In [ ]:
from monai.transforms import (
    Compose,
    EnsureChannelFirstD,
    LoadImageD,
    ResizeD,
    ScaleIntensityRangePercentilesD,
    SelectItemsD,
    ToTensorD,
)

# This example assumes output_cls=True and output_mask=True.
# Remove "mask" from every keys= list if output_mask=False.
# Remove "cls" from ToTensorD / SelectItemsD if output_cls=False.
custom_transform = Compose(
    [
        LoadImageD(keys=["img", "mask"]),
        EnsureChannelFirstD(keys=["img", "mask"]),
        ScaleIntensityRangePercentilesD(
            keys=["img"], lower=1, upper=99, b_min=-1, b_max=1, clip=True
        ),
        ResizeD(
            keys=["img", "mask"],
            spatial_size=(256, 256),
            mode=["bilinear", "nearest"],  # nearest-neighbour for binary masks
        ),
        ToTensorD(keys=["img", "mask", "cls"]),
        SelectItemsD(keys=["img", "mask", "cls"]),  # keep only what the model needs
    ]
)

ds_custom = MyDataset(
    base_image_dir=BASE_IMAGE_DIR,
    csv_path=CSV_PATH,
    transform=custom_transform,
    cache_dir="./cache/my_dataset_256",
    output_cls=True,
    output_mask=True,
)

train_ds, val_ds = ds_custom.get_datasets(n_splits=10, num_cores=4)
print(f"img shape : {train_ds[0]['img'].shape}")  # → torch.Size([1, 256, 256])
print(f"mask shape: {train_ds[0]['mask'].shape}")  # → torch.Size([1, 256, 256])
print(f"cls shape : {train_ds[0]['cls'].shape}")  # → torch.Size([N_labels])

---

## Step 6 — Use the registry

`@register_dataset("my_dataset")` registers the class the moment the cell runs. Look it up by name with no direct import needed.

| Function | |
|---|---|
| `list_datasets()` | All registered names (built-ins + custom) |
| `resolve_dataset(name)` | Returns the class — call it like the class itself |

In [ ]:
from radharmony.registry import resolve_dataset, list_datasets

print("Registered datasets:", list_datasets())

# Retrieve by name and use exactly like a direct import
DatasetCls = resolve_dataset(
    "my_dataset"
)  # MUST CHANGE: match your @register_dataset name
train_ds, val_ds = DatasetCls(
    base_image_dir=BASE_IMAGE_DIR,
    csv_path=CSV_PATH,
    label_csv_path=LABEL_CSV_PATH,
    cache_dir=CACHE_DIR,
    output_cls=True,
).get_datasets(n_splits=10, num_cores=4)

print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")

---

## Step 7 — Add your dataset to the Gradio app

`app.py` exposes a visualiser for all registered datasets. Adding yours requires two things:

### 7.1 Write a build function

The function receives the values the user types into the UI and must return `(dataset_obj, error_str | None)`:

```python
def _build_<name>(base_dir, csv_path, extra_field, cache_dir, **flags):
    ...
    return dataset_obj, None   # or (None, "error message") to abort
```

| Parameter | UI field | Notes |
|---|---|---|
| `base_dir` | Base image directory | Required |
| `csv_path` | CSV path | Pass `csv_path or None` to your dataset |
| `extra_field` | Dataset-specific field | Controlled by `extra_field_label` in the registry entry; pass `extra_field or None` |
| `cache_dir` | Cache dir | Pass `cache_dir or None` |
| `**flags` | Output checkboxes | Dict of `output_cls`, `output_mask`, `output_report`, `output_bbox` booleans |

### 7.2 Add one entry to `DATASET_REGISTRY`

```python
DATASET_REGISTRY: dict[str, DatasetConfig] = {
    ...
    "MyDataset": DatasetConfig(
        is_3d=False,                   # True for CT volumes
        extra_field_label="Label CSV", # label shown above the extra textbox; "" to hide it
        build=_build_mydataset,
    ),
}
```

That is all — the UI updates automatically.

### Example

```python
# In app.py — import your dataset class at the top
from radharmony.dataset import MyDataset

# Write the build function
def _build_mydataset(base_dir, csv_path, extra_field, cache_dir, **flags):
    return MyDataset(
        base_image_dir=base_dir,
        csv_path=csv_path or None,
        label_csv_path=extra_field or None,
        cache_dir=cache_dir or None,
        output_cls=flags.get("output_cls", False),
        output_mask=flags.get("output_mask", False),
    ), None

# Register it
DATASET_REGISTRY["MyDataset"] = DatasetConfig(
    is_3d=False,
    extra_field_label="Label CSV (optional)",
    build=_build_mydataset,
)
```
